# Away-Game Travel App: build the dataset

This notebook builds every table that `TravelData` and the Supabase database need:

| File | What it holds | Where the values come from |
|---|---|---|
| `venues.csv` | Home base + away venues | Real coordinates you enter |
| `hotels.csv` | Hotels near the away venue | **Real** names and locations from OpenStreetMap; price, rooms and amenities are **mock** |
| `restaurants.csv` | Restaurants near the venue | **Real** names, locations and cuisine from OpenStreetMap; capacity is **mock** |
| `menu_items.csv` | Menu per restaurant, with diet tags | **Mock**, generated from the cuisine type |
| `bus_options.csv` | Charter bus companies | **Mock** (fictional companies) |
| `teams.csv`, `profiles.csv` | Your team, coaches and players | **Mock** (fictional people) |
| `travel_data.json` | All of the above, nested | For `TravelData.load_data()` |

**Why the mix:** no free dataset has hotel prices, live room counts or restaurant menus. The trick is to use real places so the map and distances are believable, and mock only the fields you can't get.

**If OpenStreetMap can't be reached** (no internet, rate limit), the notebook switches to fully mock places around the venue, so it always finishes.

**Rules for OpenStreetMap data:**
- Run it once, when you build the dataset. The app reads the saved files and never calls Overpass live.
- Set your own `USER_AGENT` in cell 1.
- Show "© OpenStreetMap contributors" in the app (the data is ODbL licensed).

## 1. Settings: edit these

In [ ]:
from pathlib import Path

SEED = 42                      # same seed = same mock values every run

HOME_BASE = {                  # where the bus leaves from
    "venue_id": "V_HOME", "name": "Doyt L. Perry Stadium (BGSU)",
    "address": "1610 Stadium Dr, Bowling Green, OH 43403",
    "lat": 41.37806, "lng": -83.62250, "capacity": 24000,
}
AWAY_VENUES = [                # add more away venues here; each gets its own hotels and restaurants
    {"venue_id": "V001", "name": "Ohio Stadium",
     "address": "411 Woody Hayes Dr, Columbus, OH 43210",
     "lat": 40.00167, "lng": -83.01972, "capacity": 102780},
    {"venue_id": "V002", "name": "Glass Bowl (University of Toledo)",
     "address": "3044 N. Glass Bowl Dr, Toledo, OH 43606",
     "lat": 41.65694, "lng": -83.61361, "capacity": 26038},
    {"venue_id": "V003", "name": "Dix Stadium (Kent State)",
     "address": "2227 Summit Rd, Kent, OH 44240",
     "lat": 41.13917, "lng": -81.31333, "capacity": 25319},
    {"venue_id": "V004", "name": "Michigan Stadium",
     "address": "1201 S Main St, Ann Arbor, MI 48104",
     "lat": 42.26587, "lng": -83.74871, "capacity": 107601},
]

HOTEL_RADIUS_KM = 12           # search radius around each away venue
RESTAURANT_RADIUS_KM = 6
MAX_HOTELS = 12                # keep the nearest N per venue
MAX_RESTAURANTS = 15

SPORT = "soccer"               # soccer | basketball | football
N_PLAYERS = 24
TEAM_NAME = "BGSU Falcons"

USE_OSM = True                 # False = fully mock places, no internet needed
USER_AGENT = "AwayGameTravelApp/0.1 (hackathon project; contact: your-email@example.com)"  # put your email here

OUT = Path("catalog_csv"); RAW = OUT / "raw"   # run this notebook from the data/ folder
OUT.mkdir(exist_ok=True); RAW.mkdir(exist_ok=True)

## 2. Imports and helpers

In [ ]:
import json, math, random
import numpy as np
import pandas as pd
import requests

rng = random.Random(SEED)
np.random.seed(SEED)

def haversine_km(lat1, lng1, lat2, lng2):
    # straight-line distance on the Earth's surface
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lng2 - lng1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def est_drive_min(km, city=True):
    # rough estimate: roads are ~30% longer than a straight line; city ~30 km/h, highway ~90 km/h
    speed = 30 if city else 90
    return int(round(km * 1.3 / speed * 60))

def offset_point(lat, lng, max_km):
    # random point within max_km of (lat, lng), used only for mock places
    d = max_km * math.sqrt(rng.random()); b = rng.uniform(0, 2 * math.pi)
    dlat = (d * math.cos(b)) / 111.0
    dlng = (d * math.sin(b)) / (111.0 * math.cos(math.radians(lat)))
    return round(lat + dlat, 6), round(lng + dlng, 6)

def pg_array(items):
    # Postgres array literal so Supabase can import list columns from CSV
    return "{" + ",".join(items) + "}"

print("helpers ready;", {v["name"]: round(haversine_km(HOME_BASE["lat"], HOME_BASE["lng"], v["lat"], v["lng"])) for v in AWAY_VENUES}, "km from home (straight line)")

## 3. Fetch real hotels and restaurants from OpenStreetMap (cached)

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def fetch_osm(venue):
    cache = RAW / f"osm_{venue['venue_id']}.json"
    if cache.exists():                                  # never query twice for the same venue
        print("using cached", cache); return json.loads(cache.read_text())
    if not USE_OSM:
        return None
    q = f'''
    [out:json][timeout:60];
    (
      nwr["tourism"~"^(hotel|motel)$"](around:{HOTEL_RADIUS_KM*1000},{venue['lat']},{venue['lng']});
      nwr["amenity"="restaurant"](around:{RESTAURANT_RADIUS_KM*1000},{venue['lat']},{venue['lng']});
    );
    out center tags;
    '''
    try:
        r = requests.post(OVERPASS_URL, data={"data": q},
                          headers={"User-Agent": USER_AGENT}, timeout=90)
        r.raise_for_status()
        data = r.json()
        cache.write_text(json.dumps(data))
        print(f"OSM: {len(data.get('elements', []))} elements for {venue['name']}")
        return data
    except Exception as e:
        print(f"OSM not reachable ({type(e).__name__}): using mock places for {venue['name']}")
        return None

def parse_osm(data, venue):
    hotels, restaurants = [], []
    for el in data.get("elements", []):
        t = el.get("tags", {})
        name = t.get("name")
        lat = el.get("lat") or el.get("center", {}).get("lat")
        lng = el.get("lon") or el.get("center", {}).get("lon")
        if not name or lat is None or lng is None:
            continue                                    # skip unnamed or unlocated places
        addr = " ".join(x for x in [t.get("addr:housenumber"), t.get("addr:street")] if x)
        city = t.get("addr:city", "")
        row = {"name": name, "lat": round(lat, 6), "lng": round(lng, 6),
               "address": ", ".join(x for x in [addr, city] if x) or None,
               "osm_id": f"{el['type']}/{el['id']}", "source": "osm"}
        if t.get("tourism") in ("hotel", "motel"):
            row["stars_tag"] = t.get("stars"); hotels.append(row)
        elif t.get("amenity") == "restaurant":
            row["cuisine_tag"] = t.get("cuisine", ""); restaurants.append(row)
    return hotels, restaurants

osm = {v["venue_id"]: fetch_osm(v) for v in AWAY_VENUES}

## 4. Mock places (used when OSM is off or unreachable)

In [ ]:
HOTEL_WORDS = (["Riverside", "Campus View", "Parkway", "Lakeshore", "Union Square", "Maple Grove",
                "Gateway", "Northgate", "Heritage", "Summit", "Harbor Point", "Cedar Ridge"],
               ["Inn", "Suites", "Hotel", "Lodge", "Inn & Suites", "Grand Hotel"])
MOCK_RESTAURANTS = [("Stadium Grill", "american"), ("Bella Pasta", "italian"), ("Taqueria Sol", "mexican"),
    ("Golden Wok", "chinese"), ("Spice Route", "indian"), ("Olive Branch", "mediterranean"),
    ("Smokehouse 42", "bbq"), ("Slice Shop", "pizza"), ("Bangkok Kitchen", "thai"),
    ("Sakura House", "japanese"), ("Corner Deli", "sandwich"), ("Harvest Table", "american"),
    ("Casa Verde", "mexican"), ("Saffron Grill", "indian"), ("Trattoria Roma", "italian")]

def mock_places(venue):
    used, hotels = set(), []
    while len(hotels) < MAX_HOTELS:
        name = f"{rng.choice(HOTEL_WORDS[0])} {rng.choice(HOTEL_WORDS[1])}"
        if name in used: continue
        used.add(name); lat, lng = offset_point(venue["lat"], venue["lng"], HOTEL_RADIUS_KM)
        hotels.append({"name": name, "lat": lat, "lng": lng, "address": None,
                       "osm_id": None, "source": "mock", "stars_tag": None})
    restaurants = []
    for name, cuisine in MOCK_RESTAURANTS[:MAX_RESTAURANTS]:
        lat, lng = offset_point(venue["lat"], venue["lng"], RESTAURANT_RADIUS_KM)
        restaurants.append({"name": name, "lat": lat, "lng": lng, "address": None,
                            "osm_id": None, "source": "mock", "cuisine_tag": cuisine})
    return hotels, restaurants

## 5. Hotels: add price, rooms, amenities and distance to the venue

In [ ]:
AMENITIES = ["free_wifi", "breakfast_included", "parking_for_buses", "fitness_center",
             "meeting_room", "laundry", "pool", "late_checkout"]
CANCEL = ["Free cancellation up to 48h before check-in",
          "Free cancellation up to 7 days before check-in",
          "Non-refundable (10% cheaper)"]

def nearest(rows, venue, n):
    for r in rows:
        r["km_to_venue"] = round(haversine_km(venue["lat"], venue["lng"], r["lat"], r["lng"]), 2)
    rows = sorted(rows, key=lambda r: r["km_to_venue"])
    seen, out = set(), []
    for r in rows:                                     # OSM sometimes has duplicates of one place
        if r["name"].lower() in seen: continue
        seen.add(r["name"].lower()); out.append(r)
    return out[:n]

def enrich_hotel(h, venue, i):
    try: stars = int(float(h.get("stars_tag")))
    except (TypeError, ValueError): stars = rng.choices([2, 3, 4], weights=[3, 5, 2])[0]
    stars = min(max(stars, 1), 5)
    base = {1: 70, 2: 95, 3: 135, 4: 185, 5: 260}[stars]
    near_premium = max(0, 25 - 3 * h["km_to_venue"])   # closer to the stadium costs a bit more
    price = round(base + near_premium + rng.uniform(-12, 18), 2)
    cancel = rng.choice(CANCEL)
    if cancel.startswith("Non-refundable"): price = round(price * 0.9, 2)
    amen = sorted(rng.sample(AMENITIES, k=rng.randint(3, 6)))
    if stars >= 3 and "meeting_room" not in amen: amen.append("meeting_room")  # teams need film-session space
    return {
        "hotel_id": f"H{venue['venue_id'][1:]}{i:02d}", "venue_id": venue["venue_id"],
        "name": h["name"], "address": h["address"] or f"Near {venue['name']}",
        "lat": h["lat"], "lng": h["lng"], "stars": stars,
        "price_per_room_night": price, "room_capacity": 2,
        "rooms_available": rng.randint(8, 60), "amenities": sorted(amen),
        "cancellation_policy": cancel, "km_to_venue": h["km_to_venue"],
        "drive_min_to_venue": max(3, est_drive_min(h["km_to_venue"])),
        "source": h["source"], "osm_id": h["osm_id"],
    }

## 6. Restaurants and menus (with diet tags)

In [ ]:
# (item, price USD, diet tags, course). Halal is set per restaurant, below.
MENUS = {
 "american": [("Grilled Chicken Plate", 15.5, ["gluten_free"], "main"), ("Classic Cheeseburger", 14.0, [], "main"),
              ("Veggie Burger", 13.5, ["vegetarian"], "main"), ("Steak & Potatoes", 24.0, ["gluten_free"], "main"),
              ("Garden Salad", 9.0, ["vegetarian", "vegan", "gluten_free"], "side"), ("Mac & Cheese", 7.5, ["vegetarian"], "side"),
              ("Baked Sweet Potato", 6.0, ["vegetarian", "vegan", "gluten_free"], "side"), ("Brownie", 6.0, ["vegetarian"], "dessert")],
 "italian":  [("Spaghetti Bolognese", 16.0, [], "main"), ("Chicken Parmesan", 18.5, [], "main"),
              ("Penne Primavera", 14.5, ["vegetarian"], "main"), ("Gluten-Free Penne Marinara", 14.0, ["vegetarian", "vegan", "gluten_free"], "main"),
              ("Grilled Salmon & Vegetables", 22.0, ["gluten_free"], "main"), ("Caesar Salad", 9.5, [], "side"),
              ("Garlic Bread", 5.0, ["vegetarian"], "side"), ("Tiramisu", 8.0, ["vegetarian"], "dessert")],
 "mexican":  [("Chicken Burrito Bowl", 13.5, ["gluten_free"], "main"), ("Carne Asada Tacos", 14.0, ["gluten_free"], "main"),
              ("Black Bean Burrito", 12.0, ["vegetarian", "vegan"], "main"), ("Veggie Fajitas", 13.0, ["vegetarian", "vegan", "gluten_free"], "main"),
              ("Chips & Guacamole", 7.0, ["vegetarian", "vegan", "gluten_free"], "side"), ("Mexican Rice & Beans", 5.0, ["vegetarian", "vegan", "gluten_free"], "side"),
              ("Churros", 6.0, ["vegetarian"], "dessert")],
 "chinese":  [("Kung Pao Chicken", 14.0, [], "main"), ("Beef & Broccoli", 15.5, [], "main"),
              ("Vegetable Lo Mein", 12.5, ["vegetarian", "vegan"], "main"), ("Steamed Tofu & Vegetables", 12.0, ["vegetarian", "vegan", "gluten_free"], "main"),
              ("Steamed Rice", 3.0, ["vegetarian", "vegan", "gluten_free"], "side"), ("Vegetable Spring Rolls", 6.0, ["vegetarian", "vegan"], "side")],
 "indian":   [("Chicken Tikka Masala", 16.5, ["gluten_free"], "main"), ("Lamb Rogan Josh", 18.0, ["gluten_free"], "main"),
              ("Chana Masala", 13.0, ["vegetarian", "vegan", "gluten_free"], "main"), ("Paneer Butter Masala", 14.5, ["vegetarian", "gluten_free"], "main"),
              ("Basmati Rice", 4.0, ["vegetarian", "vegan", "gluten_free"], "side"), ("Garlic Naan", 4.0, ["vegetarian"], "side"),
              ("Mango Lassi", 5.0, ["vegetarian", "gluten_free"], "drink")],
 "mediterranean": [("Chicken Shawarma Plate", 15.0, ["gluten_free"], "main"), ("Lamb Kofta Plate", 17.0, ["gluten_free"], "main"),
              ("Falafel Plate", 13.0, ["vegetarian", "vegan"], "main"), ("Grilled Halloumi Bowl", 14.0, ["vegetarian", "gluten_free"], "main"),
              ("Hummus & Pita", 7.0, ["vegetarian", "vegan"], "side"), ("Tabbouleh", 6.5, ["vegetarian", "vegan"], "side"),
              ("Baklava", 6.0, ["vegetarian"], "dessert")],
 "bbq":      [("Pulled Pork Platter", 17.0, ["gluten_free"], "main"), ("Brisket Platter", 21.0, ["gluten_free"], "main"),
              ("Smoked Chicken Half", 16.0, ["gluten_free"], "main"), ("Smoked Portobello Sandwich", 13.0, ["vegetarian"], "main"),
              ("Coleslaw", 4.0, ["vegetarian", "gluten_free"], "side"), ("Corn on the Cob", 4.0, ["vegetarian", "vegan", "gluten_free"], "side"),
              ("Peach Cobbler", 6.5, ["vegetarian"], "dessert")],
 "pizza":    [("Pepperoni Pizza (large)", 19.0, [], "main"), ("Margherita Pizza (large)", 17.0, ["vegetarian"], "main"),
              ("Veggie Pizza on Cauliflower Crust", 20.0, ["vegetarian", "gluten_free"], "main"), ("Chicken Alfredo", 15.0, [], "main"),
              ("Side Salad", 6.0, ["vegetarian", "vegan", "gluten_free"], "side"), ("Breadsticks", 6.0, ["vegetarian"], "side")],
 "thai":     [("Chicken Pad Thai", 14.5, ["gluten_free"], "main"), ("Green Curry with Beef", 16.0, ["gluten_free"], "main"),
              ("Tofu Red Curry", 13.5, ["vegetarian", "vegan", "gluten_free"], "main"), ("Vegetable Fried Rice", 12.0, ["vegetarian"], "main"),
              ("Jasmine Rice", 3.0, ["vegetarian", "vegan", "gluten_free"], "side"), ("Fresh Spring Rolls", 7.0, ["vegetarian", "vegan", "gluten_free"], "side")],
 "japanese": [("Chicken Teriyaki Bowl", 14.5, [], "main"), ("Salmon Rice Bowl", 17.0, ["gluten_free"], "main"),
              ("Vegetable Udon", 13.0, ["vegetarian", "vegan"], "main"), ("Avocado Cucumber Roll", 8.0, ["vegetarian", "vegan", "gluten_free"], "main"),
              ("Edamame", 5.0, ["vegetarian", "vegan", "gluten_free"], "side"), ("Miso Soup", 3.5, ["vegetarian"], "side")],
 "sandwich": [("Turkey Club", 12.0, [], "main"), ("Grilled Chicken Wrap", 11.5, [], "main"),
              ("Caprese Sandwich", 10.5, ["vegetarian"], "main"), ("Hummus Veggie Wrap", 10.0, ["vegetarian", "vegan"], "main"),
              ("Fruit Cup", 4.5, ["vegetarian", "vegan", "gluten_free"], "side"), ("Kettle Chips", 2.5, ["vegetarian", "vegan", "gluten_free"], "side")],
}
CUISINE_MAP = {"burger": "american", "steak_house": "american", "diner": "american", "regional": "american",
               "barbecue": "bbq", "tex-mex": "mexican", "lebanese": "mediterranean", "greek": "mediterranean",
               "middle_eastern": "mediterranean", "turkish": "mediterranean", "sushi": "japanese", "ramen": "japanese",
               "asian": "chinese", "vietnamese": "thai", "deli": "sandwich", "sandwich": "sandwich",
               "chicken": "american", "seafood": "american", "pakistani": "indian", "nepalese": "indian"}

def norm_cuisine(tag):
    first = (tag or "").split(";")[0].strip().lower()
    return first if first in MENUS else CUISINE_MAP.get(first, "american")

def enrich_restaurant(r, venue, i):
    cuisine = norm_cuisine(r.get("cuisine_tag"))
    rid = f"R{venue['venue_id'][1:]}{i:02d}"
    halal = cuisine in ("mediterranean", "indian") or rng.random() < 0.15
    items = []
    for j, (name, price, tags, course) in enumerate(MENUS[cuisine], start=1):
        tags = list(tags)
        if halal or "vegetarian" in tags:               # meat-free items are treated as halal-compatible
            tags.append("halal")
        if "vegan" in tags or "gluten_free" in tags or rng.random() < 0.5:
            tags.append("nut_free")
        items.append({"item_id": f"{rid}-M{j:02d}", "restaurant_id": rid, "name": name, "course": course,
                      "price": round(price * rng.uniform(0.95, 1.1), 2), "diet_tags": sorted(set(tags))})
    mains = [it["price"] for it in items if it["course"] == "main"]
    return {
        "restaurant_id": rid, "venue_id": venue["venue_id"], "name": r["name"],
        "address": r["address"] or f"Near {venue['name']}", "lat": r["lat"], "lng": r["lng"],
        "cuisine": cuisine, "halal_certified": halal,
        "group_capacity": rng.choice([24, 30, 40, 50, 60, 80, 120]),
        "takes_group_reservations": rng.random() < 0.85,
        "price_per_head": round(np.mean(mains) + 6, 2),   # main + side/drink estimate
        "km_to_venue": r["km_to_venue"], "source": r["source"], "osm_id": r["osm_id"],
    }, items

## 7. Build the hotel, restaurant and menu tables

In [ ]:
hotels, restaurants, menu_items = [], [], []
for v in AWAY_VENUES:
    data = osm.get(v["venue_id"])
    h_raw, r_raw = parse_osm(data, v) if data else ([], [])
    if len(h_raw) < 3 or len(r_raw) < 3:               # too few real places: top up with mock ones
        mh, mr = mock_places(v)
        h_raw += mh if len(h_raw) < 3 else []
        r_raw += mr if len(r_raw) < 3 else []
    for i, h in enumerate(nearest(h_raw, v, MAX_HOTELS), start=1):
        hotels.append(enrich_hotel(h, v, i))
    for i, r in enumerate(nearest(r_raw, v, MAX_RESTAURANTS), start=1):
        row, items = enrich_restaurant(r, v, i)
        restaurants.append(row); menu_items += items

hotels_df = pd.DataFrame(hotels)
restaurants_df = pd.DataFrame(restaurants)
menu_df = pd.DataFrame(menu_items)
print(len(hotels_df), "hotels |", len(restaurants_df), "restaurants |", len(menu_df), "menu items")
hotels_df[["name", "stars", "price_per_room_night", "rooms_available", "km_to_venue", "source"]].head(8)

## 8. Bus companies (mock)

In [ ]:
km_home_to_venue = {v["venue_id"]: round(haversine_km(HOME_BASE["lat"], HOME_BASE["lng"], v["lat"], v["lng"]), 1)
                    for v in AWAY_VENUES}
bus_df = pd.DataFrame([
    # rate_per_km is charged on the round trip; a multi-night trip also pays for the driver's room
    {"bus_option_id": "B01", "company": "Great Lakes Coachways", "vehicle": "Motorcoach", "seats": 56,
     "rate_per_km": 2.45, "min_charge": 1100.0, "driver_overnight_fee": 180.0, "wifi": True, "luggage_bay": True},
    {"bus_option_id": "B02", "company": "Buckeye Charter Lines", "vehicle": "Motorcoach", "seats": 56,
     "rate_per_km": 2.15, "min_charge": 950.0, "driver_overnight_fee": 200.0, "wifi": False, "luggage_bay": True},
    {"bus_option_id": "B03", "company": "Maumee Valley Transit", "vehicle": "Mini coach", "seats": 35,
     "rate_per_km": 1.80, "min_charge": 750.0, "driver_overnight_fee": 160.0, "wifi": True, "luggage_bay": True},
    {"bus_option_id": "B04", "company": "Falcon Shuttle Co.", "vehicle": "Executive minibus", "seats": 24,
     "rate_per_km": 1.55, "min_charge": 600.0, "driver_overnight_fee": 150.0, "wifi": True, "luggage_bay": False},
])
bus_df

## 9. Team, coaches and players (mock people)

In [ ]:
POSITIONS = {
    "soccer": ["GK"] * 3 + ["DF"] * 8 + ["MF"] * 8 + ["FW"] * 5,
    "basketball": ["PG", "SG", "SF", "PF", "C"] * 3,
    "football": ["QB"] * 3 + ["RB"] * 4 + ["WR"] * 6 + ["TE"] * 3 + ["OL"] * 9 + ["DL"] * 8 + ["LB"] * 6 + ["DB"] * 8 + ["K", "P"],
}
FIRST = ["Aiden", "Marcus", "Diego", "Kwame", "Ethan", "Rahul", "Noah", "Omar", "Lucas", "Jamal", "Mateo", "Ryan",
         "Samir", "Tyler", "Kenji", "Andre", "Liam", "Yusuf", "Caleb", "Arjun", "Malik", "Owen", "Felipe", "Isaac",
         "Jonah", "Tariq", "Elijah", "Hassan", "Gabriel", "Nikhil"]
LAST = ["Carter", "Nguyen", "Okafor", "Silva", "Patel", "Brooks", "Haddad", "Kowalski", "Reyes", "Thompson",
        "Kim", "Mensah", "Fischer", "Ali", "Morales", "Chen", "Walker", "Rahman", "Bennett", "Sato"]
DIETS = [("vegetarian", 0.15), ("halal", 0.10), ("gluten_free", 0.06), ("nut_free", 0.05), ("vegan", 0.03)]

team_df = pd.DataFrame([{"team_id": "T01", "name": TEAM_NAME, "sport": SPORT,
                         "home_venue_id": HOME_BASE["venue_id"], "join_code": "FALCON-" + str(rng.randint(1000, 9999))}])
users, names = [], set()
def new_name():
    while True:
        n = (rng.choice(FIRST), rng.choice(LAST))
        if n not in names: names.add(n); return n

for k, title in enumerate(["Head Coach", "Assistant Coach", "Athletic Trainer"], start=1):
    f, l = new_name()
    users.append({"user_id": f"U-C{k:02d}", "team_id": "T01", "role": "coach", "name": f"{f} {l}",
                  "email": f"{f}.{l}.coach@example.edu".lower(), "title": title,
                  "jersey_no": None, "position": None, "diet_tags": []})
positions = (POSITIONS[SPORT] * 5)[:N_PLAYERS]
jerseys = rng.sample(range(1, 99), N_PLAYERS)
for k in range(N_PLAYERS):
    f, l = new_name()
    diet = [d for d, p in DIETS if rng.random() < p]
    if "vegan" in diet and "vegetarian" not in diet: diet.append("vegetarian")
    users.append({"user_id": f"U-P{k+1:02d}", "team_id": "T01", "role": "player", "name": f"{f} {l}",
                  "email": f"{f}.{l}{k+1}@example.edu".lower(), "title": None,
                  "jersey_no": jerseys[k], "position": positions[k], "diet_tags": sorted(diet)})
users_df = pd.DataFrame(users)
users_df["jersey_no"] = users_df["jersey_no"].astype("Int64")   # whole numbers, blank for coaches
print(users_df.role.value_counts().to_dict(), "| players with a diet need:",
      int((users_df.diet_tags.str.len() > 0).sum()))
users_df.head(6)

## 10. Check that the dataset can actually support a trip

In [ ]:
travelers = len(users_df)
rooms_needed = math.ceil(len(users_df[users_df.role == "player"]) / 2) + len(users_df[users_df.role == "coach"])  # coaches get singles
problems = []

assert hotels_df.hotel_id.is_unique and restaurants_df.restaurant_id.is_unique and menu_df.item_id.is_unique
assert set(menu_df.restaurant_id) == set(restaurants_df.restaurant_id), "every restaurant needs a menu"

fits = hotels_df[hotels_df.rooms_available >= rooms_needed]
if fits.empty: problems.append(f"no single hotel has {rooms_needed} rooms")
group_ok = restaurants_df[(restaurants_df.group_capacity >= travelers) & restaurants_df.takes_group_reservations]
if group_ok.empty: problems.append(f"no restaurant seats {travelers}")

needed_tags = set(t for tags in users_df.diet_tags for t in tags)
for rid, grp in menu_df.groupby("restaurant_id"):
    have = set(t for tags in grp.diet_tags for t in tags)
    if not needed_tags <= have:
        problems.append(f"{rid} has no option for {sorted(needed_tags - have)}")

print(f"travelers: {travelers} | rooms needed: {rooms_needed} | hotels that fit: {len(fits)} | "
      f"restaurants that seat the team: {len(group_ok)}")
print("home -> venue:", km_home_to_venue, "km straight line")
print("\n".join(problems) if problems else "all checks passed")

## 11. Save: CSVs for Supabase, JSON for `TravelData.load_data()`

In [ ]:
venues_df = pd.DataFrame([HOME_BASE] + AWAY_VENUES)

def to_csv(df, name, list_cols=()):
    out = df.copy()
    for c in list_cols:
        out[c] = out[c].apply(pg_array)              # {a,b} format imports into text[] columns
    out.to_csv(OUT / name, index=False)

to_csv(venues_df, "venues.csv")
to_csv(hotels_df, "hotels.csv", ["amenities"])
to_csv(restaurants_df, "restaurants.csv")
to_csv(menu_df, "menu_items.csv", ["diet_tags"])
to_csv(bus_df, "bus_options.csv")
to_csv(team_df, "teams.csv")
to_csv(users_df, "profiles.csv", ["diet_tags"])

# nested version: each restaurant carries its menu, ready for the app
def records(df): return json.loads(df.to_json(orient="records"))
rest = records(restaurants_df)
for r in rest:
    r["menu"] = records(menu_df[menu_df.restaurant_id == r["restaurant_id"]].drop(columns="restaurant_id"))
travel_data = {
    "meta": {"seed": SEED, "home_base": HOME_BASE["venue_id"],
             "km_home_to_venue": km_home_to_venue,
             "attribution": "Place names and locations © OpenStreetMap contributors (ODbL), where source = osm",
             "mock_fields": "prices, rooms_available, amenities, menus, bus companies and people are mock data"},
    "venues": records(venues_df), "hotels": records(hotels_df), "restaurants": rest,
    "bus_options": records(bus_df), "teams": records(team_df), "users": records(users_df),
}
(OUT / "travel_data.json").write_text(json.dumps(travel_data, indent=2))
print("saved:", sorted(p.name for p in OUT.glob("*.*")))

## 12. Load it in the app

```python
class TravelData:
    def load_data(self, path="data/travel_data.json"):
        d = json.loads(Path(path).read_text())
        self.venues = {v["venue_id"]: v for v in d["venues"]}
        self.hotels, self.restaurants, self.bus_rates = d["hotels"], d["restaurants"], d["bus_options"]

    def hotels_near(self, venue_id, radius_km=10):
        return sorted((h for h in self.hotels if h["venue_id"] == venue_id and h["km_to_venue"] <= radius_km),
                      key=lambda h: h["km_to_venue"])

    def get_menu(self, restaurant_id, diet_tags=()):
        r = next(r for r in self.restaurants if r["restaurant_id"] == restaurant_id)
        return [m for m in r["menu"] if set(diet_tags) <= set(m["diet_tags"])]
```

**Supabase:** run `python scripts/csv_to_seed_sql.py data/catalog_csv > supabase/seed/catalog_seed_osm.sql` from the `travel-planner` folder, then paste that file into the Supabase SQL editor. The catalog tables are created by `supabase/migrations/`.

In [ ]:
# quick test of the loader above
d = json.loads((OUT / "travel_data.json").read_text())
r0 = d["restaurants"][0]
print(r0["name"], f"({r0['cuisine']}, {r0['km_to_venue']} km from venue)")
print("halal + gluten-free options:", [m["name"] for m in r0["menu"] if {"halal", "gluten_free"} <= set(m["diet_tags"])])